In [ ]:
import json
import pandas as pd
from pathlib import Path
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Initialize Geocoder
geolocator = Nominatim(user_agent="mission_event_geocoder_final")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# Define Directory Paths
raw_dir = Path("../Data/raw/events")
club_dir = Path("../Data/processed/clubs")
event_dir = Path("../Data/processed/events")
div_dir = Path("../Data/processed/divisions")

# Ensure processed directories exist
for folder in [club_dir, event_dir, div_dir]:
    folder.mkdir(parents=True, exist_ok=True)

def get_physical_location(location_str):
    """Attempt to find a standardized address. Return None if not found."""
    try:
        if not location_str: return None
        result = geocode(location_str)
        return result.address if result else None
    except Exception:
        return None

def get_recommendation(row):
    """Generates the audit recommendation for club records."""
    if row['IsPlaceholder'] and row['MultipleIDsFound']:
        return "REPLACE: Placeholder ID exists for a verified name"
    elif row['IsPlaceholder']:
        return "VERIFY: Placeholder ID with no verified match found"
    elif row['MultipleIDsFound']:
        return "REVIEW: Multiple verified IDs found for same name"
    else:
        return "VALID: Unique verified record"

def process_mission_data():
    events, clubs, divisions, participations = [], [], [], []

    print("Extracting data from raw JSON files...")
    for file_path in raw_dir.glob("*.json"):
        with open(file_path, 'r') as f:
            data = json.load(f)
            e_id = data.get("EventId")
            
            # Extract Event Info
            events.append({
                "EventId": e_id,
                "Name": data.get("Name"),
                "StartDate": pd.to_datetime(data.get("StartDate")),
                "EndDate": pd.to_datetime(data.get("EndDate")),
                "Location": data.get("Location"),
                "IsOver": data.get("IsOver")
            })
            
            # Extract Club/Participation/Division Info
            for club in data.get("Clubs", []):
                clubs.append({"ClubId": club.get("ClubId"), "Name": club.get("Name")})
                participations.append({"EventId": e_id, "ClubId": club.get("ClubId")})
                
            for div in data.get("Divisions", []):
                divisions.append({
                    "DivisionId": div.get("DivisionId"), "EventId": e_id,
                    "Name": div.get("Name"), "TeamCount": div.get("TeamCount"),
                    "CodeAlias": div.get("CodeAlias")
                })

    # --- 1. Process Events ---
    df_events = pd.DataFrame(events).drop_duplicates(subset=["EventId"])
    df_events = df_events.sort_values(by="StartDate").reset_index(drop=True)
    df_events["StartDate"] = df_events["StartDate"].dt.strftime('%Y-%m-%d')
    df_events["EndDate"] = df_events["EndDate"].dt.strftime('%Y-%m-%d')
    
    print("Resolving physical addresses (this will take ~1s per unique location)...")
    unique_locations = df_events["Location"].unique()
    location_map = {loc: get_physical_location(loc) for loc in unique_locations}
    df_events["PhysicalAddress"] = df_events["Location"].map(location_map)

    # --- 2. Process Clubs with Audit Metadata ---
    df_clubs = pd.DataFrame(clubs).drop_duplicates(subset=["ClubId"])
    
    # Add normalization column for comparison
    df_clubs['NormName'] = df_clubs['Name'].astype(str).str.strip().str.upper()
    df_clubs['IsPlaceholder'] = df_clubs['ClubId'] < 0
    name_counts = df_clubs.groupby('NormName')['ClubId'].transform('count')
    df_clubs['MultipleIDsFound'] = name_counts > 1
    df_clubs['ActionRecommendation'] = df_clubs.apply(get_recommendation, axis=1)
    
    # Sort by normalized name to group duplicates together visually
    df_clubs = df_clubs.sort_values(by=['NormName', 'ClubId']).reset_index(drop=True)
    df_clubs = df_clubs.drop(columns=['NormName']) # Remove temp column

    # --- 3. Process Participations and Divisions ---
    df_participations = pd.DataFrame(participations).drop_duplicates()
    df_divisions = pd.DataFrame(divisions).drop_duplicates(subset=["DivisionId"])

    # --- 4. Save All Files ---
    df_events.to_csv(event_dir / "events_master.csv", index=False)
    df_clubs.to_csv(club_dir / "clubs_master.csv", index=False)
    df_participations.to_csv(club_dir / "event_participations.csv", index=False)
    df_divisions.to_csv(div_dir / "divisions_master.csv", index=False)

    print(f"✅ Mission complete.")
    print(f"   - Processed {len(df_events)} events.")
    print(f"   - Generated audited club registry with {len(df_clubs)} unique IDs.")

if __name__ == "__main__":
    process_mission_data()
